## Comparison between all different Models ##

In [ ]:
import time
import numpy as np
import cupy as cp
import torch
import mne
import sys
import os

sys.path.append("..")
import act_var1 as act_cpu
import act_gpu_cuda as act_cupy
import act_gpu_pytorch as act_torch

In [ ]:
# ---------------- SETTINGS ----------------
fs = 256
epoch_length = 3 * 256
order = 10

num_epochs = 51
num_repeats = 5

eeg_channels = ["HB_1"]

fc_info = (0.5, 15, 0.5)
logDt_info = (-4, 1, 0.5)
c_info = (-10, 10, 0.5)

tc_res = epoch_length
tc_info = (0, tc_res, 32)

# ---------------- LOAD EEG ----------------
print("Loading EEG...")
edf_path = os.path.join(os.getcwd(), '../Testing Scripts/Data/sub-1_task-Sleep_acq-headband_eeg.edf')
raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
raw.pick_channels(eeg_channels)
raw.notch_filter(freqs=50)
raw.filter(l_freq=0.1, h_freq=20.0, verbose=False)
data = raw.get_data().T

Loading EEG...
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
Filtering raw data in 1 contiguous segment
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 1691 samples (6.605 s)



In [3]:
# ------------- CREATE EPOCHS ----------------
segments_cpu = []
total_epochs = data.shape[0] // epoch_length
for i in range(total_epochs):
    start = i * epoch_length
    end = start + epoch_length
    segments_cpu.append(data[start:end, 0].astype(np.float32))
segments_cpu = segments_cpu[:num_epochs]

segments_cupy  = [cp.asarray(s) for s in segments_cpu]
segments_torch = [torch.tensor(s, device="cuda") for s in segments_cpu]

print("Epochs used:", len(segments_cpu))

Epochs used: 51


In [4]:
# ---------------- BENCHMARK FUNCTION ----------------
def run_test(name, module, segments, hybrid):
    print(f"\n{'='*40}\nRunning: {name}\n{'='*40}")

    all_epoch_times = []
    all_full_times  = []
    all_norm        = []

    for repeat in range(num_repeats):
        print(f"  repeat {repeat+1}/{num_repeats}")

        t_full_start = time.perf_counter()

        if name == "cpu":
            act_obj = module.ACT(
                FS=fs, length=epoch_length, tc_info=tc_info,
                fc_info=fc_info, logDt_info=logDt_info,
                c_info=c_info, force_regenerate=True, mute=True
            )
        elif name in ["cupy_gpu", "cupy_hybrid"]:
            act_obj = module.ACT(
                FS=fs, length=epoch_length, tc_info=tc_info,
                fc_info=fc_info, logDt_info=logDt_info,
                c_info=c_info, hybrid=hybrid, force_regenerate=True, mute=True
            )
        elif name in ["torch_gpu", "torch_hybrid"]:
            mode = "hybrid" if hybrid else "gpu"
            act_obj = module.ACT(
                FS=fs, length=epoch_length, tc_info=tc_info,
                fc_info=fc_info, logDt_info=logDt_info,
                c_info=c_info, mode=mode, force_regenerate=True, mute=True
            )
        epoch_times = []
        norm_vals   = []

        for i, seg in enumerate(segments):
            t0  = time.perf_counter()
            out = act_obj.transform(seg, order=order, debug=False)

            if isinstance(seg, cp.ndarray):
                cp.cuda.Stream.null.synchronize()
            elif torch.is_tensor(seg):
                torch.cuda.synchronize()

            elapsed = time.perf_counter() - t0
            
            norm = out["norm_residue"]
            if isinstance(norm, cp.ndarray):
                norm = norm.get()

            if i > 0:  # skip first epoch as warmup
                epoch_times.append(elapsed)
                norm_vals.append(float(np.squeeze(norm)))

        all_epoch_times.append(epoch_times)
        all_full_times.append(time.perf_counter() - t_full_start)
        all_norm.append(norm_vals)

    return all_epoch_times, all_full_times, all_norm

In [5]:
# ---------------- RUN ----------------
results = {}
results["cpu"]          = run_test("cpu",          act_cpu,   segments_cpu,   False)
results["cupy_gpu"]     = run_test("cupy_gpu",     act_cupy,  segments_cupy,  False)
results["cupy_hybrid"]  = run_test("cupy_hybrid",  act_cupy,  segments_cupy,  True)
results["torch_gpu"]    = run_test("torch_gpu",    act_torch, segments_torch, False)
results["torch_hybrid"] = run_test("torch_hybrid", act_torch, segments_torch, True)


Running: cpu
  repeat 1/5
Dictionary length: 278400
  repeat 2/5
Dictionary length: 278400
  repeat 3/5
Dictionary length: 278400
  repeat 4/5
Dictionary length: 278400
  repeat 5/5
Dictionary length: 278400

Running: cupy_gpu
  repeat 1/5
CPU cores: 32
GPU Devices detected: 1
Hybrid mode: False
Unified memory: False
Dictionary length: 278400
Generating dictionary on GPU...
Dictionary Generated.
  repeat 2/5
CPU cores: 32
GPU Devices detected: 1
Hybrid mode: False
Unified memory: False
Dictionary length: 278400
Generating dictionary on GPU...
Dictionary Generated.
  repeat 3/5
CPU cores: 32
GPU Devices detected: 1
Hybrid mode: False
Unified memory: False
Dictionary length: 278400
Generating dictionary on GPU...
Dictionary Generated.
  repeat 4/5
CPU cores: 32
GPU Devices detected: 1
Hybrid mode: False
Unified memory: False
Dictionary length: 278400
Generating dictionary on GPU...
Dictionary Generated.
  repeat 5/5
CPU cores: 32
GPU Devices detected: 1
Hybrid mode: False
Unified memory

In [6]:
W = 80
print(f"\n{'='*W}")
print(f"{'IMPLEMENTATION':<16} {'EPOCH (s)':<24} {'FULL RUN (s)':<20} {'NORM RESIDUE'}")
print(f"{'-'*W}")

for name, (all_epoch_times, all_full_times, all_norm) in results.items():
    flat_epochs = [t for rep in all_epoch_times for t in rep]
    flat_norm   = [v for rep in all_norm        for v in rep]

    epoch_mean = np.mean(flat_epochs)
    epoch_std  = np.std(flat_epochs)
    full_mean  = np.mean(all_full_times)
    full_std   = np.std(all_full_times)
    norm_mean  = np.mean(flat_norm)
    norm_std   = np.std(flat_norm)

    print(
        f"{name:<16} "
        f"{epoch_mean:.6f} ± {epoch_std:.6f}  "
        f"{full_mean:.3f} ± {full_std:.3f}       "
        f"{norm_mean:.4f} ± {norm_std:.4f}"
    )

print(f"{'='*W}")


IMPLEMENTATION   EPOCH (s)                FULL RUN (s)         NORM RESIDUE
--------------------------------------------------------------------------------
cpu              0.166333 ± 0.023144  15.342 ± 0.238       0.1255 ± 0.0236
cupy_gpu         0.025047 ± 0.004686  63.374 ± 0.431       0.1255 ± 0.0236
cupy_hybrid      0.025129 ± 0.004667  8.372 ± 0.120       0.1255 ± 0.0236
torch_gpu        0.029391 ± 0.016090  29.419 ± 0.300       0.1245 ± 0.0239
torch_hybrid     0.021781 ± 0.004752  7.952 ± 0.016       0.1256 ± 0.0236
